<a href="https://colab.research.google.com/github/YujiaLIAO-1/housing/blob/main/10_12_2024_mis_session2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd

df = pd.read_csv('train.csv')
df.info()

# Prepare the data by separating X and y and dropping unimportant features
# Only 7 features used: 'Age', 'SibSp', 'Fare', 'Parch', 'Sex', 'Embarked', 'Pclass'
X = df.drop(['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
y = df['Survived']
X.info()

# Split the data into a training set and a test set.
# Any number for the random_state is fine, see 42: https://en.wikipedia.org/wiki/42_(number)
# We choose to use 20% (test_size=0.2) of the data set as the test set.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-

In [18]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 331 to 102
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    712 non-null    int64  
 1   Sex       712 non-null    object 
 2   Age       572 non-null    float64
 3   SibSp     712 non-null    int64  
 4   Parch     712 non-null    int64  
 5   Fare      712 non-null    float64
 6   Embarked  710 non-null    object 
dtypes: float64(2), int64(3), object(2)
memory usage: 44.5+ KB


In [19]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

# Split the data into a training set and a test set.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [20]:
# select sub dataframe based on data type
df_num = X.select_dtypes(exclude ='object')
df_cat = X.select_dtypes(include ='object')

# we need the column names as lists
num_features = df_num.columns.tolist()
cat_features = df_cat.columns.tolist()

In [21]:
num_features = ['Age', 'SibSp', 'Fare', 'Parch']
cat_features = ['Sex', 'Embarked', 'Pclass']

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# NOTE the step names can be arbitrary

# Create the preprocessing pipeline for numerical features
# Step 1 filling the missing values if any using mean
# Step 2 is standardization using the z-score

# 对数字进行标准化，对文字进行onehot encoding
num_pipeline = Pipeline(
    steps=[
        ('num_imputer', SimpleImputer()),
        ('scaler', StandardScaler()),
        ]
)

# Create the preprocessing pipelines for the categorical features
# Step 1: filling the missing values if any using the most frequent value
# Step 2: one hot encoding

cat_pipeline = Pipeline(
    steps=[
        ('cat_imputer', SimpleImputer()),
        ('onehot', OneHotEncoder()),
    ]
)

In [23]:
# 合并
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, num_features),
        ('cat_pipeline', cat_pipeline, cat_features),
    ]
)

In [24]:
# Specify the model to use, which is DecisionTreeClassifier in this example
# Make a full pipeline by combining preprocessor and the model

from sklearn.tree import DecisionTreeClassifier

# final decision tree (dt) pipeline
dt_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('tree_clf', DecisionTreeClassifier()),
    ]
)

In [25]:
# GridSearch with 10-fold cross validation and accuracy as the metric
from sklearn.model_selection import GridSearchCV

param_grid = [
    {
        'preprocessor__num_pipeline__num_imputer__strategy': ['mean', 'median'],
        'preprocessor__cat_pipeline__cat_imputer__strategy': ['most_frequent'],  # only one choice for this parameter
        'tree_clf__criterion': ['gini', 'entropy'],
        'tree_clf__max_depth': [3, 7, 10, 12, 15],
    }
]
#两个下划线：去找里面的东西
# 2*1*2*5=20 个组合

# set up the grid search
grid_search = GridSearchCV(dt_pipeline, param_grid, cv=10, scoring='accuracy')

In [26]:
# train the model using the full pipeline
grid_search.fit(X_train, y_train)

# check the best performing parameter combination
grid_search.best_params_


{'preprocessor__cat_pipeline__cat_imputer__strategy': 'most_frequent',
 'preprocessor__num_pipeline__num_imputer__strategy': 'mean',
 'tree_clf__criterion': 'entropy',
 'tree_clf__max_depth': 3}

In [27]:
grid_search.cv_results_['mean_test_score']
# 训练了20*5cv=100个模型

array([0.82443271, 0.80191706, 0.78660016, 0.78372457, 0.7654734 ,
       0.82584116, 0.79630282, 0.79219484, 0.78231612, 0.78515258,
       0.82443271, 0.80056729, 0.78517214, 0.77672144, 0.78100548,
       0.82584116, 0.79489437, 0.78796948, 0.77951878, 0.76834898])

In [28]:
grid_search.cv_results_['mean_test_score'].max()

0.8258411580594679

In [29]:
tree_clf_best = grid_search.best_estimator_

In [31]:
# final evaluation using the test data
y_pred = tree_clf_best.predict(X_test)

# calculate accuracy, precision, recall, f1-score
# y_test is the ground truth, y_pred is our model's prediction
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

print(f'Accuracy Score : {accuracy_score(y_test, y_pred)}')
print(f'Precision Score : {precision_score(y_test, y_pred)}')
print(f'Recall Score : {recall_score(y_test, y_pred)}')
print(f'F1 Score : {f1_score(y_test, y_pred)}')


Accuracy Score : 0.7988826815642458
Precision Score : 0.796875
Recall Score : 0.6891891891891891
F1 Score : 0.7391304347826086


In [32]:
titanic_test = pd.read_csv('test.csv')
titanic_test.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.0+ KB


In [33]:
y_pred_titanic = tree_clf_best.predict(titanic_test)

In [34]:
# combine id and prediction for kaggle submission
dt_pipeline_submit = pd.DataFrame({
    'PassengerId': titanic_test['PassengerId'],
    'Survived': y_pred_titanic
})


dt_pipeline_submit


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [35]:

# generate the csv
dt_pipeline_submit.to_csv('dt-pipeline-submit.csv', index=False)